# 70. 비라벨 가스 시드 근접표본 준지도 앙상블
외부GitHub의소수불량시드근접표본아이디어를참고해학습가스/정상거리비교로비라벨약한라벨을생성한다. 각내부·외부폴드에서teacher를다시학습하고전체620라벨키및평가기간±100분을비라벨후보에서제외한다. 학습기간밖비라벨도사용하지않는다. pseudo50불량/100정상상한,약한가중치. 평가는진짜라벨만사용한다.
이미본개발자료중첩검증이며새표본독립검증아니다. pseudo라벨을실제불량발견으로세지않는다. 원래저장소의숫자재현이아닌아이디어차용이다.


In [ ]:
from pathlib import Path
import json,numpy as np,pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.metrics import confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
s68=''.join(json.loads((root/'notebooks/68_rg3_history_normalized_cluster_experts.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
exec(s68.split('tail=s67')[0].replace('rolling(100,min_periods=20)','rolling(100,min_periods=5)').replace('rg3_history_ratio_','rg3_pseudoseed_'))
U=observed[observed['product'].eq('RG3')&(observed.Max_Injection_Pressure>0)&(observed.Barrel_Temperature_5>0)].copy()
U=U[~pd.MultiIndex.from_frame(U[keys]).isin(pd.MultiIndex.from_frame(d[keys]))].reset_index(drop=True)
ux={}
for sensor in ['Barrel_Temperature_2','Hopper_Temperature','Cushion_Position','Barrel_Temperature_5']:
 den=U[sensor+'__past100_std_new'].where(U[sensor+'__past100_std_new']>1e-8)
 ux[sensor+'__jitter10_over100']=U[sensor+'__past10_std']/den
 ux[sensor+'__delta_over100']=U[sensor+'__delta1']/den
UF=pd.DataFrame(ux).replace([np.inf,-np.inf],np.nan)
memo={};audit=[]
def predict(ref,q):
 key=(tuple(ref),tuple(q))
 if key in memo:return memo[key]
 if len(np.unique(y[ref]))<2:return np.zeros((len(q),3))
 assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
 im=SimpleImputer(strategy='median',keep_empty_features=True).fit(F.iloc[ref][y[ref]==0])
 scale=RobustScaler().fit(im.transform(F.iloc[ref][y[ref]==0]))
 tr=np.clip(scale.transform(im.transform(F.iloc[ref])),-10,10);te=np.clip(scale.transform(im.transform(F.iloc[q])),-10,10)
 keep=U.TimeStamp.between(d.TimeStamp.iloc[ref].min(),d.TimeStamp.iloc[ref].max())
 # Remove query period plus rolling-history buffer from pseudo candidate selection.
 keep&=~U.TimeStamp.between(d.TimeStamp.iloc[q].min()-pd.Timedelta(minutes=100),d.TimeStamp.iloc[q].max()+pd.Timedelta(minutes=100))
 ids=np.flatnonzero(keep)
 if len(ids)>5000:ids=np.random.default_rng(42).choice(ids,5000,replace=False)
 pu=np.clip(scale.transform(im.transform(UF.iloc[ids])),-10,10)
 pos=tr[y[ref]==1];normal=tr[y[ref]==0]
 nn=NearestNeighbors(n_neighbors=min(3,len(normal))).fit(normal)
 pn=NearestNeighbors(n_neighbors=1).fit(pos)
 def teacher(a):
  dp=pn.kneighbors(a)[0].mean(axis=1);dn=nn.kneighbors(a)[0].mean(axis=1)
  return dn/(dn+dp+1e-8)
 t=teacher(pu) if len(pu) else np.array([])
 pi=np.flatnonzero(t>.7);ni=np.flatnonzero(t<.2)
 pi=pi[np.argsort(t[pi])[::-1]][:50];ni=ni[np.argsort(t[ni])][:100]
 xx=np.vstack([tr,pu[pi],pu[ni]])
 yy=np.r_[y[ref],np.ones(len(pi),int),np.zeros(len(ni),int)]
 weight=np.r_[np.ones(len(ref)),np.full(len(pi),.25),np.full(len(ni),.1)]
 out0=[];out1=[]
 for cls in [RandomForestClassifier,ExtraTreesClassifier]:
  for seed in [42,7]:
   m=cls(n_estimators=100,max_depth=5,min_samples_leaf=3,class_weight='balanced',random_state=seed,n_jobs=2)
   out0.append(m.fit(tr,y[ref]).predict_proba(te)[:,1])
   m=cls(n_estimators=100,max_depth=5,min_samples_leaf=3,class_weight='balanced',random_state=seed,n_jobs=2)
   out1.append(m.fit(xx,yy,sample_weight=weight).predict_proba(te)[:,1])
 memo[key]=np.column_stack([np.mean(out0,axis=0),np.mean(out1,axis=0),teacher(te)])
 audit.append(dict(train=len(ref),query=len(q),candidate_unlabeled=len(ids),pseudo_positive=len(pi),pseudo_negative=len(ni),query_start=str(d.TimeStamp.iloc[q].min()),query_end=str(d.TimeStamp.iloc[q].max())))
 return memo[key]
records=[];choices=[]
for b in blocks:
 q=np.flatnonzero(groups.eq(b));ref=np.flatnonzero(~groups.eq(b))
 out=predict(ref,q);inner=np.full((len(ref),3),np.nan)
 for ib in groups.iloc[ref].unique():
  mask=groups.iloc[ref].eq(ib).to_numpy();inner[mask]=predict(ref[~mask],ref[mask])
 candidates=[]
 for k,name in enumerate(['supervised_control','pseudo_seed_ensemble','nearest_gas_normal_ratio']):
  for budget in [.02,.05,.1]:
   cut=float(np.quantile(inner[y[ref]==0,k],1-budget,method='higher'))
   pr=inner[:,k]>cut
   tn,fp,fn,tp=confusion_matrix(y[ref],pr,labels=[0,1]).ravel()
   recall=tp/(tp+fn) if tp+fn else 0;f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0
   candidates.append((recall,f1,-fp,name,budget,out[:,k]>cut))
   for j,i in enumerate(q):records.append(dict(model=name,budget=budget,TimeStamp=d.TimeStamp.iloc[i],block=int(b),actual=int(y[i]),prediction=int(out[j,k]>cut),score=float(out[j,k]),threshold=cut))
 chosen=max(candidates,key=lambda a:a[:3]);choices.append(dict(block=int(b),selected=chosen[3],budget=chosen[4],inner_recall=chosen[0]))
 for j,i in enumerate(q):records.append(dict(model='inner_recall_first_selected',budget=0,TimeStamp=d.TimeStamp.iloc[i],block=int(b),actual=int(y[i]),prediction=int(chosen[5][j]),score=np.nan,threshold=np.nan))
 print('Finished fold-local pseudo-seed block',b,flush=True)
p=pd.DataFrame(records);rows=[]
for (name,budget),g in p.groupby(['model','budget']):
 tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
 rows.append(dict(model=name,budget=budget,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),recall=tp/(tp+fn),F1=2*tp/(2*tp+fp+fn)))
m=pd.DataFrame(rows)
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(audit),'pseudo_audit'),(pd.DataFrame(choices),'inner_choices')]:
 obj.to_csv(tables/f'rg3_foldlocal_pseudoseed_{name}.csv',index=False,encoding='utf-8-sig')
print(m.to_string(index=False))
print('Pseudo labels are weak training hypotheses, NEVER evaluation ground truth.')
